In [1]:
import re
from pathlib import Path

import pandas as pd
from scipy.stats import ttest_ind

In [2]:
version = "eval_v2"
clfs = {"attn"}

paths = sorted(Path("output/input_space_v2").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
state_table = pd.concat(tables, ignore_index=True)
print(state_table.shape)
state_table.head()

48
(168, 20)


,key,space,base_lr,run,model,repr,clf,dataset,ckpt,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,11,0.001110,0.05,32,"[3.7, 1.0]",train,0.001176,0.999842,0.000099,0.999847,0.000099
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,11,0.001110,0.05,32,"[3.7, 1.0]",validation,0.046283,0.987599,0.001759,0.986927,0.002051
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,11,0.001110,0.05,32,"[3.7, 1.0]",test,0.062541,0.986905,0.001479,0.984041,0.002004
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,best,6,0.000216,0.05,22,"[0.72, 1.0]",train,2.149933,0.355973,0.002436,0.299480,0.002486
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,best,6,0.000216,0.05,22,"[0.72, 1.0]",validation,2.399023,0.283684,0.005492,0.225597,0.005170


In [3]:
state_summary = state_table.loc[state_table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "run", "repr", "clf"], columns="dataset"
)
state_summary = state_summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
state_summary

acc                   acc_std             
dataset                 hcpya_task21 nsd_cococlip hcpya_task21 nsd_cococlip
space       base_lr run                                                    
flat        0.0010  1       0.986905     0.304267     0.001479     0.005561
                    2       0.984722     0.288126     0.001714     0.005232
                    3       0.987500     0.304824     0.001575     0.005490
                    4       0.985119     0.300186     0.001665     0.005522
                    5       0.986905     0.283488     0.001573     0.005049
                    6       0.986111     0.310390     0.001564     0.005615
                    7       0.986310     0.298516     0.001648     0.005308
                    8       0.984524     0.312430     0.001643     0.005532
mni_cortex  0.0010  1       0.958929     0.266976     0.002734     0.005045
                    2       0.955159     0.257514     0.002821     0.004845
                    3       0.957738     0.264193     0.002792     0.004977
                    4       0.958730     0.255844     0.002779     0.005090
                    5       0.960119     0.265863     0.002702     0.005181
                    6       0.956746     0.267904     0.002797     0.005154
                    7       0.958333     0.256957     0.002859     0.004853
                    8       0.956349     0.267347     0.002813     0.004948
schaefer400 0.0003  1       0.975992     0.274954     0.002195     0.005144
                    2       0.973413     0.269759     0.002184     0.004704
                    3       0.972817     0.268089     0.002229     0.005255
                    4       0.977778     0.274768     0.002090     0.005595
                    5       0.975992     0.281818     0.002216     0.005443
                    6       0.978373     0.274583     0.002010     0.004939
                    7       0.970437     0.267347     0.002395     0.005284
                    8       0.974603     0.263451     0.002247     0.005192

In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [5]:
SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}

SPACE_LRS = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

In [6]:
state_datasets = ["hcpya_task21", "nsd_cococlip"]

records = []

for (space, base_lr, run), row in state_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in state_datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   run | HCP-YA Task21   | NSD COCO24   |
|:------------|-------:|------:|:----------------|:-------------|
| flat        | 0.001  |     1 | 98.7 ± 0.1      | 30.4 ± 0.6   |
| flat        | 0.001  |     2 | 98.5 ± 0.2      | 28.8 ± 0.5   |
| flat        | 0.001  |     3 | 98.8 ± 0.2      | 30.5 ± 0.5   |
| flat        | 0.001  |     4 | 98.5 ± 0.2      | 30.0 ± 0.6   |
| flat        | 0.001  |     5 | 98.7 ± 0.2      | 28.3 ± 0.5   |
| flat        | 0.001  |     6 | 98.6 ± 0.2      | 31.0 ± 0.6   |
| flat        | 0.001  |     7 | 98.6 ± 0.2      | 29.9 ± 0.5   |
| flat        | 0.001  |     8 | 98.5 ± 0.2      | 31.2 ± 0.6   |
| mni_cortex  | 0.001  |     1 | 95.9 ± 0.3      | 26.7 ± 0.5   |
| mni_cortex  | 0.001  |     2 | 95.5 ± 0.3      | 25.8 ± 0.5   |
| mni_cortex  | 0.001  |     3 | 95.8 ± 0.3      | 26.4 ± 0.5   |
| mni_cortex  | 0.001  |     4 | 95.9 ± 0.3      | 25.6 ± 0.5   |
| mni_cortex  | 0.001  |     5 | 96.0 ± 0.3      | 26.6 ± 0.5   |
| mni_cort

In [7]:
records = []
for space, name in SPACE_NAMES.items():
    record = {"space": name}
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space   | HCP-YA Task21   | NSD COCO24   |
|:--------|:----------------|:-------------|
| parcel  | 97.5 ± 0.3      | 27.2 ± 0.6   |
| flat    | 98.6 ± 0.1      | 30.0 ± 1.0   |
| volume  | 95.8 ± 0.2      | 26.3 ± 0.5   |
\begin{tabular}{lll}
\toprule
space & HCP-YA Task21 & NSD COCO24 \\
\midrule
parcel & 97.5 $\mypm$ 0.3 & 27.2 $\mypm$ 0.6 \\
flat & 98.6 $\mypm$ 0.1 & 30.0 $\mypm$ 1.0 \\
volume & 95.8 $\mypm$ 0.2 & 26.3 $\mypm$ 0.5 \\
\bottomrule
\end{tabular}



In [8]:
spaces = ["schaefer400", "flat", "mni_cortex"]

test_results = {}

for ds in state_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = state_summary.loc[(space_a, SPACE_LRS[space_a]), ("acc", ds)].values
            acc_b = state_summary.loc[(space_b, SPACE_LRS[space_b]), ("acc", ds)].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

hcpya_task21 schaefer400 flat t=-10.9 p=1.2e-06
hcpya_task21 schaefer400 mni_cortex t=15.7 p=4.1e-09
hcpya_task21 flat mni_cortex t=41.0 p=1.2e-14
nsd_cococlip schaefer400 flat t=-6.9 p=2.5e-05
nsd_cococlip schaefer400 mni_cortex t=3.3 p=5.4e-03
nsd_cococlip flat mni_cortex t=9.3 p=2.3e-06


In [9]:
version = "eval_v2"
clfs = {"logistic"}

paths = sorted(Path("output/input_space_v2").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
trait_table = pd.concat(tables, ignore_index=True)
print(trait_table.shape)
trait_table.head()

336
(67872, 17)


,key,space,base_lr,run,model,repr,clf,dataset,trial,C,split,acc,acc_std,f1,f1_std,bacc,bacc_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,cls,logistic,aabc_age,NaN,0.046416,train,0.872047,0.014516,0.870560,0.014763,0.871340,0.014544
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,cls,logistic,aabc_age,NaN,0.046416,test,0.365385,0.067426,0.361781,0.067032,0.360348,0.066803
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,cls,logistic,aabc_age,1.0,0.046416,train,0.868110,0.014870,0.867895,0.014936,0.869526,0.014760
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,cls,logistic,aabc_age,1.0,0.046416,test,0.384615,0.064105,0.371388,0.059713,0.379350,0.063051
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,cls,logistic,aabc_age,2.0,0.359381,train,0.998031,0.001960,0.998093,0.001897,0.998201,0.001791


In [10]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


trait_repr = "patch"

trait_summary = trait_table.query("split == 'test' and trial > 0").pivot_table(
    values=["acc", "bacc"],
    index=["space", "base_lr", "run", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
trait_summary = trait_summary.loc[
    (slice(None), slice(None), slice(None), trait_repr, "logistic")
].dropna(axis=1, how="all")
trait_summary

mean                                 \
                              acc                                  
dataset                  aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr run                                            
flat        0.0010  1    0.427115  0.873818  0.611129   0.598769   
                    2    0.460000  0.863273  0.619597   0.591846   
                    3    0.455769  0.880000  0.628468   0.597846   
                    4    0.437115  0.865636  0.609194   0.602154   
                    5    0.433462  0.879636  0.631774   0.594308   
                    6    0.446923  0.850727  0.619355   0.599846   
                    7    0.450769  0.870000  0.610565   0.600923   
                    8    0.440577  0.868000  0.622903   0.596308   
mni_cortex  0.0010  1    0.532075  0.869310  0.628790   0.598308   
                    2    0.536792  0.868276  0.613387   0.584000   
                    3    0.521321  0.875172  0.612581   0.595538   
                    4    0.514528  0.863103  0.621290   0.597692   
                    5    0.531132  0.871897  0.622903   0.600923   
                    6    0.535472  0.858793  0.607903   0.598615   
                    7    0.531698  0.880345  0.625806   0.588923   
                    8    0.531509  0.870345  0.613065   0.600308   
schaefer400 0.0003  1    0.440192  0.726727  0.634274   0.594000   
                    2    0.449423  0.739636  0.625726   0.590462   
                    3    0.441731  0.707455  0.633065   0.573077   
                    4    0.443269  0.729091  0.640968   0.610615   
                    5    0.425962  0.716909  0.619677   0.584308   
                    6    0.429038  0.717091  0.624677   0.581231   
                    7    0.432885  0.698727  0.616694   0.585231   
                    8    0.439615  0.715636  0.645000   0.580923   

                                                                              \
                                                                        bacc   
dataset                 adni_ad_vs_cn hcpya_rest1lr_gender ppmi_dx  aabc_age   
space       base_lr run                                                        
flat        0.0010  1        0.759024             0.869808  0.6313  0.424799   
                    2        0.760244             0.857308  0.6348  0.457296   
                    3        0.756098             0.873077  0.6373  0.453567   
                    4        0.755122             0.876442  0.6407  0.434380   
                    5        0.752439             0.874808  0.6466  0.430288   
                    6        0.752683             0.874327  0.6459  0.444352   
                    7        0.743902             0.849327  0.6214  0.448791   
                    8        0.747073             0.870481  0.6271  0.437589   
mni_cortex  0.0010  1        0.777317             0.909327  0.6213  0.534368   
                    2        0.732439             0.917500  0.6288  0.539162   
                    3        0.748293             0.909038  0.6201  0.523448   
                    4        0.751951             0.917500  0.6166  0.517074   
                    5        0.732683             0.910769  0.6341  0.533132   
                    6        0.755854             0.917212  0.6156  0.537624   
                    7        0.740244             0.917115  0.6294  0.533462   
                    8        0.744878             0.915577  0.6322  0.533668   
schaefer400 0.0003  1        0.740732             0.813750  0.6556  0.437287   
                    2        0.721951             0.809038  0.6563  0.448107   
                    3        0.714390             0.809519  0.6549  0.438583   
                    4        0.734634             0.815962  0.6527  0.440916   
                    5        0.722683             0.807500  0.6468  0.423807   
                    6        0.725122             0.810962  0.6486  0.426435   
                    7        0.712683       

In [11]:
trait_datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex"]
trait_metric = "bacc"

records = []

for (space, base_lr, run), row in trait_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in trait_datasets:
        acc = row[("mean", trait_metric, ds)]
        std = row[("sem", trait_metric, ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   run | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   |
|:------------|-------:|------:|:-----------|:-------------|:-----------|:-----------|:------------|:------------|
| flat        | 0.001  |     1 | 60.5 ± 0.4 | 58.2 ± 0.5   | 65.3 ± 0.8 | 59.4 ± 0.4 | 42.5 ± 0.6  | 86.7 ± 0.4  |
| flat        | 0.001  |     2 | 61.6 ± 0.5 | 57.5 ± 0.6   | 65.9 ± 0.8 | 58.7 ± 0.5 | 45.7 ± 0.7  | 85.6 ± 0.4  |
| flat        | 0.001  |     3 | 62.4 ± 0.4 | 58.4 ± 0.6   | 66.5 ± 0.8 | 59.4 ± 0.4 | 45.4 ± 0.5  | 87.4 ± 0.5  |
| flat        | 0.001  |     4 | 60.3 ± 0.4 | 58.6 ± 0.5   | 64.5 ± 0.8 | 60.9 ± 0.4 | 43.4 ± 0.6  | 85.8 ± 0.5  |
| flat        | 0.001  |     5 | 62.6 ± 0.4 | 57.7 ± 0.5   | 63.2 ± 0.8 | 61.6 ± 0.4 | 43.0 ± 0.6  | 87.4 ± 0.5  |
| flat        | 0.001  |     6 | 61.4 ± 0.5 | 58.5 ± 0.5   | 63.8 ± 0.8 | 59.8 ± 0.5 | 44.4 ± 0.7  | 84.4 ± 0.5  |
| flat        | 0.001  |     7 | 60.4 ± 0.4 | 58.5 ± 0.6   | 62.7 ± 1.0 | 57.5 ±

In [12]:
for ds in trait_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = trait_summary.loc[
                (space_a, SPACE_LRS[space_a]), ("mean", trait_metric, ds)
            ].values
            acc_b = trait_summary.loc[
                (space_b, SPACE_LRS[space_b]), ("mean", trait_metric, ds)
            ].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

abide_dx schaefer400 flat t=2.3 p=3.8e-02
abide_dx schaefer400 mni_cortex t=2.9 p=1.4e-02
abide_dx flat mni_cortex t=0.4 p=6.8e-01
adhd200_dx schaefer400 flat t=-2.5 p=3.5e-02
adhd200_dx schaefer400 mni_cortex t=-2.0 p=7.1e-02
adhd200_dx flat mni_cortex t=0.6 p=5.5e-01
adni_ad_vs_cn schaefer400 flat t=-5.9 p=7.4e-05
adni_ad_vs_cn schaefer400 mni_cortex t=-3.8 p=1.9e-03
adni_ad_vs_cn flat mni_cortex t=0.8 p=4.2e-01
ppmi_dx schaefer400 flat t=3.3 p=7.7e-03
ppmi_dx schaefer400 mni_cortex t=6.9 p=9.7e-06
ppmi_dx flat mni_cortex t=1.7 p=1.2e-01
aabc_age schaefer400 flat t=-1.2 p=2.5e-01
aabc_age schaefer400 mni_cortex t=-25.0 p=5.4e-13
aabc_age flat mni_cortex t=-18.6 p=3.3e-10
aabc_sex schaefer400 flat t=-26.4 p=1.1e-12
aabc_sex schaefer400 mni_cortex t=-29.2 p=8.7e-12
aabc_sex flat mni_cortex t=-0.5 p=6.4e-01


In [13]:
records = []

for space in ["schaefer400", "mni_cortex", "flat"]:
    record = {"space": SPACE_NAMES[space]}
    for ds in trait_datasets:
        acc = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].mean()
        std = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space   | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   | HCP-YA Task21   | NSD COCO24   |
|:--------|:-----------|:-------------|:-----------|:-----------|:------------|:------------|:----------------|:-------------|
| parcel  | 62.4 ± 1.0 | 57.0 ± 1.3   | 59.4 ± 2.1 | 61.2 ± 0.6 | 43.5 ± 0.8  | 70.8 ± 1.3  | 97.5 ± 0.3      | 27.2 ± 0.6   |
| volume  | 61.2 ± 0.7 | 58.0 ± 0.7   | 63.7 ± 2.4 | 58.6 ± 0.8 | 53.1 ± 0.7  | 86.4 ± 0.7  | 95.8 ± 0.2      | 26.3 ± 0.5   |
| flat    | 61.3 ± 0.9 | 58.2 ± 0.4   | 64.5 ± 1.3 | 59.5 ± 1.3 | 44.1 ± 1.2  | 86.2 ± 1.0  | 98.6 ± 0.1      | 30.0 ± 1.0   |
\begin{tabular}{lllllllll}
\toprule
space & ABIDE Dx & ADHD200 Dx & ADNI Dx & PPMI Dx & HCP-A Age & HCP-A Sex & HCP-YA Task21 & NSD COCO24 \\
\midrule
parcel & 62.4 $\mypm$ 1.0 & 57.0 $\mypm$ 1.3 & 59.4 $\mypm$ 2.1 & 61.2 $\mypm$ 0.6 & 43.5 $\mypm$ 0.8 & 70.8 $\mypm$ 1.3 & 97.5 $\mypm$ 0.3 & 27.2 $\mypm$ 0.6 \\
volume & 61.2 $\mypm$ 0.7 & 58.0 $\mypm$ 0.7 & 63.7 

In [14]:
test_table = pd.DataFrame.from_dict(test_results, orient="index")
test_table

statistic        pvalue
schaefer400 flat       hcpya_task21  -10.906147  1.243982e-06
            mni_cortex hcpya_task21   15.659246  4.083665e-09
flat        mni_cortex hcpya_task21   41.012485  1.178302e-14
schaefer400 flat       nsd_cococlip   -6.890375  2.474990e-05
            mni_cortex nsd_cococlip    3.293115  5.417690e-03
flat        mni_cortex nsd_cococlip    9.316070  2.283833e-06
schaefer400 flat       abide_dx        2.299483  3.768643e-02
            mni_cortex abide_dx        2.860910  1.354161e-02
flat        mni_cortex abide_dx        0.427260  6.758937e-01
schaefer400 flat       adhd200_dx     -2.514109  3.465622e-02
            mni_cortex adhd200_dx     -2.005445  7.125091e-02
flat        mni_cortex adhd200_dx      0.620065  5.471014e-01
schaefer400 flat       adni_ad_vs_cn  -5.934380  7.434390e-05
            mni_cortex adni_ad_vs_cn  -3.840474  1.872364e-03
flat        mni_cortex adni_ad_vs_cn   0.836663  4.210026e-01
schaefer400 flat       ppmi_dx         3.289681  7.744486e-03
            mni_cortex ppmi_dx         6.897927  9.671466e-06
flat        mni_cortex ppmi_dx         1.658071  1.232907e-01
schaefer400 flat       aabc_age       -1.218348  2.457583e-01
            mni_cortex aabc_age      -25.027013  5.423133e-13
flat        mni_cortex aabc_age      -18.591442  3.300444e-10
schaefer400 flat       aabc_sex      -26.425976  1.093335e-12
            mni_cortex aabc_sex      -29.248699  8.716921e-12
flat        mni_cortex aabc_sex       -0.484288  6.362768e-01

In [15]:
# cutoff = 0.001 / len(test_table)
cutoff = 1e-4
# cutoff = 0.05 / len(test_table)
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.0001


statistic        pvalue
schaefer400 flat       hcpya_task21  -10.906147  1.243982e-06
            mni_cortex hcpya_task21   15.659246  4.083665e-09
flat        mni_cortex hcpya_task21   41.012485  1.178302e-14
schaefer400 flat       nsd_cococlip   -6.890375  2.474990e-05
flat        mni_cortex nsd_cococlip    9.316070  2.283833e-06
schaefer400 flat       adni_ad_vs_cn  -5.934380  7.434390e-05
            mni_cortex ppmi_dx         6.897927  9.671466e-06
                       aabc_age      -25.027013  5.423133e-13
flat        mni_cortex aabc_age      -18.591442  3.300444e-10
schaefer400 flat       aabc_sex      -26.425976  1.093335e-12
            mni_cortex aabc_sex      -29.248699  8.716921e-12

In [16]:
cutoff = 0.05 / len(test_table)
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.002


statistic        pvalue
schaefer400 flat       hcpya_task21  -10.906147  1.243982e-06
            mni_cortex hcpya_task21   15.659246  4.083665e-09
flat        mni_cortex hcpya_task21   41.012485  1.178302e-14
schaefer400 flat       nsd_cococlip   -6.890375  2.474990e-05
flat        mni_cortex nsd_cococlip    9.316070  2.283833e-06
schaefer400 flat       adni_ad_vs_cn  -5.934380  7.434390e-05
            mni_cortex adni_ad_vs_cn  -3.840474  1.872364e-03
                       ppmi_dx         6.897927  9.671466e-06
                       aabc_age      -25.027013  5.423133e-13
flat        mni_cortex aabc_age      -18.591442  3.300444e-10
schaefer400 flat       aabc_sex      -26.425976  1.093335e-12
            mni_cortex aabc_sex      -29.248699  8.716921e-12

In [17]:
import json

sigmas = {}

space = "flat"
for ds in trait_datasets:
    sigmas[ds] = round(
        trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].std(), 4
    )
for ds in state_datasets:
    sigmas[ds] = round(state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std(), 4)
print(json.dumps(sigmas, indent=4))

{
    "abide_dx": 0.009,
    "adhd200_dx": 0.0042,
    "adni_ad_vs_cn": 0.013,
    "ppmi_dx": 0.0126,
    "aabc_age": 0.0115,
    "aabc_sex": 0.0099,
    "hcpya_task21": 0.0011,
    "nsd_cococlip": 0.0101
}
